# 04 — Interfaz: C4.5 y Random Forest

Esta interfaz permite comparar el árbol C4.5 propio con Random Forest sin tocar código.

**Cómo usarla**

1. En Colab: **Entorno de ejecución → Ejecutar todas**. No hace falta instalar nada.
2. Elegir el dataset y los parámetros de cada modelo.
3. Usar una de las tres pestañas:
   - **Comparar modelos:** entrena ambos con las mismas particiones y muestra métricas,
     gráfico y matrices de confusión.
   - **Variar un parámetro:** muestra cómo cambia la accuracy con un parámetro.
   - **Ver el árbol:** dibuja el árbol C4.5 entrenado.

Cada repetición usa otra partición (semilla, semilla + 1, ...). Los resultados muestran la
media y el desvío de esas repeticiones. Las métricas Precision, Recall y F1 son macro.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORIO = "https://github.com/marianoInsa/Decision-Tree-vs-Random-Forest.git"


def ubicar_repositorio():
    """Busca la raíz del proyecto; en Colab la clona si todavía no está."""
    for carpeta in (Path.cwd(), *Path.cwd().parents):
        if (carpeta / "datos_tp.py").is_file():
            return carpeta
    if "google.colab" not in sys.modules:
        raise FileNotFoundError("Ejecutá desde la raíz del repositorio o notebooks/.")
    destino = Path("/content/Decision-Tree-vs-Random-Forest")
    if not destino.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORIO, str(destino)], check=True)
    return destino


RAIZ = ubicar_repositorio()
os.chdir(RAIZ)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


In [ ]:
import json
from types import SimpleNamespace

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import SVG, display
from sklearn.model_selection import train_test_split

from datos_tp import cargar_datasets

# Reutiliza los modelos y el motor del notebook 03, sin copiar sus definiciones.
motor = {}
for celda in json.loads((RAIZ / "notebooks/03_experimentos.ipynb").read_text(encoding="utf-8"))["cells"]:
    if "benchmark-definiciones" in celda.get("metadata", {}).get("tags", []):
        exec(compile("".join(celda["source"]), "03_experimentos.ipynb", "exec"), motor)
ejecutar_benchmark = motor["ejecutar_benchmark"]
ArbolC45 = motor["c45"].ArbolC45

DATOS = cargar_datasets()
DATASETS = {
    "Heart Disease (mixto, 303 casos)": "heart",
    "Iris (continuo, 150 casos)": "iris",
    "Contact Lenses (discreto, 24 casos)": "lenses",
}
COLORES = {"C4.5": "#2a78d6", "Random Forest": "#eb6834"}
TINTA, TINTA_SUAVE, GRILLA, FONDO = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
METRICAS = {"accuracy": "Accuracy", "precision_macro": "Precision",
            "recall_macro": "Recall", "f1_macro": "F1"}
BARRIDOS = {
    "C4.5": {"max_depth": "1, 2, 3, 4, 5, 6, 8, 10", "min_samples_split": "2, 5, 10, 20, 40",
             "min_gain_ratio": "0, 0.02, 0.05, 0.1, 0.2", "threshold": "0, 0.01, 0.02, 0.05, 0.1"},
    "Random Forest": {"n_estimators": "10, 25, 50, 100, 200", "max_depth": "1, 2, 3, 4, 5, 6, 8, 10"},
}


def particionar_con(nombre, semilla, fraccion_test):
    """Partición estratificada; C4.5 y Random Forest reciben exactamente la misma."""
    datos = DATOS[nombre]
    return train_test_split(datos["X"], datos["y"], test_size=fraccion_test, random_state=semilla,
                            shuffle=True, stratify=datos["y"]["objetivo"])


def _repetir(nombre, grillas, repeticiones, semilla, fraccion_test):
    tablas = []
    for numero in range(repeticiones):
        actual = semilla + numero
        X_train, X_test, y_train, y_test = particionar_con(nombre, actual, fraccion_test)
        tabla = ejecutar_benchmark(
            nombre, X_train, y_train, X_test, y_test, grillas, nombre_evaluacion="test",
            semilla=actual, semilla_particion=actual,
            protocolo_particion=f"estratificada, test {fraccion_test:.0%}")
        tablas.append(tabla.assign(repeticion=numero + 1))
    return pd.concat(tablas, ignore_index=True)


def comparar(nombre, params_c45, params_rf, repeticiones=5, semilla=42, fraccion_test=0.2):
    """Devuelve una fila por modelo, repetición y conjunto evaluado."""
    grillas = {"C4.5": {k: [v] for k, v in params_c45.items()},
               "Random Forest": {k: [v] for k, v in params_rf.items()}}
    return _repetir(nombre, grillas, repeticiones, semilla, fraccion_test)


def resumir(detalle):
    """Media y desvío por modelo sobre test, más accuracy de entrenamiento y tiempos."""
    filas = {}
    for modelo, grupo in detalle.groupby("modelo", sort=False):
        test = grupo[grupo["conjunto"] == "test"]
        train = grupo[grupo["conjunto"] == "entrenamiento"]
        fila = {}
        for columna, etiqueta in METRICAS.items():
            fila[etiqueta] = test[columna].mean()
            fila[f"{etiqueta} (desvío)"] = test[columna].std(ddof=0)
        fila["Accuracy en entrenamiento"] = train["accuracy"].mean()
        fila["Entrenamiento (ms)"] = train["tiempo_entrenamiento_ms"].mean()
        fila["Predicción en test (ms)"] = test["tiempo_inferencia_ms"].mean()
        filas[modelo] = fila
    return pd.DataFrame(filas).T


def matrices_confusion(detalle):
    """Suma las matrices de test de todas las repeticiones, por modelo."""
    test = detalle[detalle["conjunto"] == "test"]
    salida = {}
    for modelo, grupo in test.groupby("modelo", sort=False):
        salida[modelo] = (np.sum([np.array(m) for m in grupo["matriz_confusion"]], axis=0),
                          list(grupo["clases"].iloc[0]))
    return salida


def barrer(nombre, modelo, parametro, valores, params_c45, params_rf,
           repeticiones=5, semilla=42, fraccion_test=0.2):
    """Varía un parámetro de un modelo; devuelve media y desvío por valor y conjunto."""
    base = dict(params_c45 if modelo == "C4.5" else params_rf)
    base.pop(parametro, None)
    grillas = {"C4.5": {"max_depth": [0]}, "Random Forest": {"n_estimators": [1]}}
    grillas[modelo] = {**{k: [v] for k, v in base.items()}, parametro: list(valores)}
    detalle = _repetir(nombre, grillas, repeticiones, semilla, fraccion_test)
    detalle = detalle[detalle["modelo"] == modelo].copy()
    detalle["valor"] = [p[parametro] for p in detalle["parametros"]]
    return (detalle.groupby(["valor", "conjunto"])["accuracy"].agg(media="mean", desvio=lambda s: s.std(ddof=0))
            .reset_index())


def entrenar_arbol(nombre, params_c45, semilla=42, fraccion_test=0.2):
    X_train, _, y_train, _ = particionar_con(nombre, semilla, fraccion_test)
    return ArbolC45(**params_c45).fit(X_train, y_train)


def medir_arbol(nodo, profundidad=0):
    """Devuelve (nodos, hojas, profundidad máxima)."""
    if nodo.es_hoja:
        return 1, 1, profundidad
    partes = [medir_arbol(hijo, profundidad + 1) for hijo in nodo.hijos.values()]
    return (1 + sum(p[0] for p in partes), sum(p[1] for p in partes), max(p[2] for p in partes))


def _rama(nodo, rama):
    if nodo.tipo == "numerico":
        return f"≤ {nodo.umbral:g}" if rama == "<=" else f"> {nodo.umbral:g}"
    return str(rama)


def arbol_dot(raiz, profundidad_maxima=4):
    """Describe el árbol en lenguaje DOT de Graphviz, hasta la profundidad pedida."""
    lineas = ['digraph arbol {', 'node [shape=box, style="rounded,filled", fontname="Helvetica", fontsize=11];',
              'edge [fontname="Helvetica", fontsize=10];']
    contador = [0]

    def texto(valor):
        return str(valor).replace("\\", "\\\\").replace('"', '\\"')

    def visitar(nodo, profundidad):
        ident = f"n{contador[0]}"
        contador[0] += 1
        n = f"n = {nodo.peso_total:g}"
        if nodo.es_hoja:
            lineas.append(f'{ident} [label="{texto(nodo.clase)}\\n{n}", fillcolor="#cfe2f8"];')
        elif profundidad >= profundidad_maxima:
            lineas.append(f'{ident} [label="… subárbol sin dibujar\\n{n}", fillcolor="#eeeeea"];')
        else:
            lineas.append(f'{ident} [label="{texto(nodo.atributo)}\\nrazón de ganancia = {nodo.gain_ratio:.3f}\\n{n}", '
                          'fillcolor="#fde3d6"];')
            for rama, hijo in nodo.hijos.items():
                destino = visitar(hijo, profundidad + 1)
                lineas.append(f'{ident} -> {destino} [label="{texto(_rama(nodo, rama))}"];')
        return ident

    visitar(raiz, 0)
    lineas.append("}")
    return "\n".join(lineas)


def arbol_texto(nodo, profundidad_maxima=4, profundidad=0, prefijo=""):
    """Alternativa en texto para un entorno sin Graphviz."""
    sangria = "    " * profundidad
    if nodo.es_hoja:
        return [f"{sangria}{prefijo}{nodo.clase} (n = {nodo.peso_total:g})"]
    if profundidad >= profundidad_maxima:
        return [f"{sangria}{prefijo}… (n = {nodo.peso_total:g})"]
    lineas = [f"{sangria}{prefijo}{nodo.atributo}"]
    for rama, hijo in nodo.hijos.items():
        lineas += arbol_texto(hijo, profundidad_maxima, profundidad + 1, f"[{_rama(nodo, rama)}] ")
    return lineas


def mostrar_arbol(raiz, profundidad_maxima):
    try:
        import graphviz
        display(SVG(graphviz.Source(arbol_dot(raiz, profundidad_maxima)).pipe(format="svg")))
    except Exception:
        print("\n".join(arbol_texto(raiz, profundidad_maxima)))


def _ejes(ax):
    ax.set_facecolor(FONDO)
    for lado in ("top", "right", "left"):
        ax.spines[lado].set_visible(False)
    ax.spines["bottom"].set_color(GRILLA)
    ax.grid(axis="y", color=GRILLA, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=TINTA_SUAVE, length=0)


def graficar_metricas(resumen):
    fig, ax = plt.subplots(figsize=(8, 3.6), facecolor=FONDO)
    etiquetas = list(METRICAS.values())
    posiciones = np.arange(len(etiquetas))
    ancho = 0.34
    for desplazamiento, modelo in zip((-0.18, 0.18), resumen.index):
        medias = [resumen.loc[modelo, m] for m in etiquetas]
        desvios = [resumen.loc[modelo, f"{m} (desvío)"] for m in etiquetas]
        barras = ax.bar(posiciones + desplazamiento, medias, ancho, yerr=desvios, label=modelo,
                        color=COLORES[modelo], error_kw={"ecolor": TINTA_SUAVE, "capsize": 3, "linewidth": 1})
        ax.bar_label(barras, labels=[f"{m:.2f}" for m in medias], padding=6, color=TINTA, fontsize=9)
    ax.set_xticks(posiciones, etiquetas)
    ax.set_ylim(0, 1.15)
    ax.set_yticks(np.arange(0, 1.01, 0.25))
    ax.set_title("Métricas en test (media y desvío)", loc="left", color=TINTA, fontsize=11)
    ax.legend(frameon=False, ncols=2, loc="upper right", labelcolor=TINTA)
    _ejes(ax)
    fig.tight_layout()
    return fig


def graficar_matrices(matrices):
    fig, ejes = plt.subplots(1, len(matrices), figsize=(4 * len(matrices), 3.6), facecolor=FONDO)
    for ax, (modelo, (matriz, clases)) in zip(np.atleast_1d(ejes), matrices.items()):
        ax.imshow(matriz, cmap="Blues", vmin=0)
        posiciones = range(len(clases))
        ax.set_xticks(posiciones, clases, rotation=20, color=TINTA_SUAVE)
        ax.set_yticks(posiciones, clases, color=TINTA_SUAVE)
        ax.set_xlabel("Predicción", color=TINTA_SUAVE)
        ax.set_ylabel("Clase real", color=TINTA_SUAVE)
        ax.set_title(modelo, loc="left", color=TINTA, fontsize=11)
        ax.tick_params(length=0)
        for lado in ax.spines.values():
            lado.set_visible(False)
        limite = matriz.max() * 0.6
        for fila in posiciones:
            for columna in posiciones:
                valor = matriz[fila, columna]
                ax.text(columna, fila, str(int(valor)), ha="center", va="center",
                        color="#ffffff" if valor > limite else TINTA)
    fig.suptitle("Matrices de confusión (suma de las repeticiones)", x=0.02, ha="left", color=TINTA, fontsize=11)
    fig.tight_layout()
    return fig


def graficar_barrido(curvas, modelo, parametro):
    fig, ax = plt.subplots(figsize=(8, 3.6), facecolor=FONDO)
    estilos = {"test": ("Test", COLORES["C4.5"], "-", "o"),
               "entrenamiento": ("Entrenamiento", COLORES["Random Forest"], "--", "s")}
    posiciones = np.arange(curvas["valor"].nunique())
    for conjunto, (etiqueta, color, linea, marca) in estilos.items():
        datos = curvas[curvas["conjunto"] == conjunto].sort_values("valor")
        ax.plot(posiciones, datos["media"], color=color, linestyle=linea, marker=marca,
                linewidth=2, markersize=6, label=etiqueta)
        ax.fill_between(posiciones, datos["media"] - datos["desvio"], datos["media"] + datos["desvio"],
                        color=color, alpha=0.15, linewidth=0)
    ax.set_xticks(posiciones, [f"{v:g}" for v in sorted(curvas["valor"].unique())])
    ax.set_xlabel(parametro, color=TINTA_SUAVE)
    ax.set_ylabel("Accuracy", color=TINTA_SUAVE)
    ax.set_title(f"{modelo}: accuracy según {parametro} (media y desvío)", loc="left", color=TINTA, fontsize=11)
    ax.legend(frameon=False, labelcolor=TINTA)
    _ejes(ax)
    fig.tight_layout()
    return fig


In [ ]:
ESTILO = {"description_width": "185px"}
ANCHO = widgets.Layout(width="400px")


def _deslizador(clase, descripcion, ayuda, **opciones):
    return clase(description=descripcion, tooltip=ayuda, style=ESTILO, layout=ANCHO,
                 continuous_update=False, **opciones)


def _limite_profundidad(valor):
    """Casilla y deslizador: sin límite desactiva el deslizador."""
    casilla = widgets.Checkbox(value=False, description="Limitar profundidad", indent=False)
    profundidad = _deslizador(widgets.IntSlider, "Profundidad máxima", "Niveles que puede crecer el árbol.",
                              value=valor, min=1, max=20, disabled=True)
    casilla.observe(lambda cambio: setattr(profundidad, "disabled", not cambio["new"]), names="value")
    return casilla, profundidad


def _caja(titulo, ayuda, hijos):
    encabezado = widgets.HTML(f"<b>{titulo}</b><br><span style='opacity:.75'>{ayuda}</span>")
    return widgets.VBox([encabezado, *hijos], layout=widgets.Layout(
        border="1px solid #d6d5d0", padding="10px", margin="0 12px 12px 0", width="440px"))


def _numeros(texto):
    valores = [float(parte) for parte in texto.replace(";", ",").split(",") if parte.strip()]
    if not valores:
        raise ValueError("escribí al menos un valor, separado por comas.")
    return [int(v) if v.is_integer() else v for v in valores]


def crear_interfaz():
    c = SimpleNamespace()
    c.dataset = widgets.Dropdown(options=DATASETS, description="Dataset", style=ESTILO, layout=ANCHO)
    c.test = _deslizador(widgets.FloatSlider, "Proporción de prueba", "Parte de los casos reservada para evaluar.",
                         value=0.2, min=0.1, max=0.4, step=0.05, readout_format=".0%")
    c.repeticiones = _deslizador(widgets.IntSlider, "Repeticiones", "Cantidad de particiones distintas.",
                                 value=5, min=1, max=30)
    c.semilla = widgets.BoundedIntText(value=42, min=0, max=100000, description="Semilla", style=ESTILO, layout=ANCHO)

    c.c45_limite, c.c45_profundidad = _limite_profundidad(5)
    c.c45_minimo = _deslizador(widgets.IntSlider, "Mínimo para dividir", "Masa mínima de un nodo para dividirlo.",
                               value=2, min=2, max=50)
    c.c45_ganancia = _deslizador(widgets.FloatSlider, "Ganancia mínima (bits)", "Por debajo no se divide.",
                                 value=0.0, min=0.0, max=0.5, step=0.01)
    c.c45_razon = _deslizador(widgets.FloatSlider, "Razón de ganancia mínima", "Por debajo no se divide.",
                              value=0.0, min=0.0, max=0.5, step=0.01)

    c.rf_arboles = _deslizador(widgets.IntSlider, "Cantidad de árboles", "Árboles del bosque.",
                               value=100, min=10, max=500, step=10)
    c.rf_atributos = widgets.Dropdown(options={"Raíz cuadrada": "sqrt", "Logaritmo en base 2": "log2", "Todos": None},
                                      description="Atributos por división", style=ESTILO, layout=ANCHO)
    c.rf_criterio = widgets.Dropdown(options={"Gini": "gini", "Entropía": "entropy"}, description="Criterio",
                                     style=ESTILO, layout=ANCHO)
    c.rf_bootstrap = widgets.Checkbox(value=True, description="Usar bootstrap", indent=False)
    c.rf_limite, c.rf_profundidad = _limite_profundidad(5)

    def params_c45():
        return {"max_depth": c.c45_profundidad.value if c.c45_limite.value else None,
                "min_samples_split": c.c45_minimo.value, "threshold": c.c45_ganancia.value,
                "min_gain_ratio": c.c45_razon.value}

    def params_rf():
        return {"n_estimators": c.rf_arboles.value, "max_features": c.rf_atributos.value,
                "criterion": c.rf_criterio.value, "bootstrap": c.rf_bootstrap.value,
                "max_depth": c.rf_profundidad.value if c.rf_limite.value else None}

    def comunes():
        return {"repeticiones": c.repeticiones.value, "semilla": c.semilla.value, "fraccion_test": c.test.value}

    estado = widgets.HTML("Elegí los parámetros y usá una pestaña.")
    salidas = {nombre: widgets.Output() for nombre in ("comparar", "barrer", "arbol")}
    botones = {}

    def boton(clave, texto, tarea):
        control = widgets.Button(description=texto, button_style="primary", icon="play",
                                 layout=widgets.Layout(width="240px"))

        def ejecutar(_):
            control.disabled = True
            estado.value = "⏳ Ejecutando…"
            salidas[clave].clear_output(wait=True)
            with salidas[clave]:
                try:
                    tarea()
                    estado.value = "✅ Listo."
                except Exception as error:
                    estado.value = f"⚠️ No se pudo ejecutar: {error}"
            control.disabled = False

        control.on_click(ejecutar)
        botones[clave] = control
        return control

    def mostrar_figura(figura):
        display(figura)
        plt.close(figura)

    def tarea_comparar():
        detalle = comparar(c.dataset.value, params_c45(), params_rf(), **comunes())
        resumen = resumir(detalle)
        columnas = [*METRICAS.values(), "Accuracy en entrenamiento", "Entrenamiento (ms)", "Predicción en test (ms)"]
        display(resumen[columnas].T.round(3))
        mostrar_figura(graficar_metricas(resumen))
        mostrar_figura(graficar_matrices(matrices_confusion(detalle)))

    c.barrido_modelo = widgets.Dropdown(options=list(BARRIDOS), description="Modelo", style=ESTILO, layout=ANCHO)
    c.barrido_parametro = widgets.Dropdown(options=list(BARRIDOS["C4.5"]), description="Parámetro", style=ESTILO, layout=ANCHO)
    c.barrido_valores = widgets.Text(value=BARRIDOS["C4.5"]["max_depth"], description="Valores (con comas)",
                                     style=ESTILO, layout=ANCHO)

    def al_cambiar_modelo(_):
        c.barrido_parametro.options = list(BARRIDOS[c.barrido_modelo.value])

    def al_cambiar_parametro(_):
        c.barrido_valores.value = BARRIDOS[c.barrido_modelo.value][c.barrido_parametro.value]

    c.barrido_modelo.observe(al_cambiar_modelo, names="value")
    c.barrido_parametro.observe(al_cambiar_parametro, names="value")

    def tarea_barrer():
        modelo, parametro = c.barrido_modelo.value, c.barrido_parametro.value
        curvas = barrer(c.dataset.value, modelo, parametro, _numeros(c.barrido_valores.value),
                        params_c45(), params_rf(), **comunes())
        mostrar_figura(graficar_barrido(curvas, modelo, parametro))
        tabla = curvas.pivot(index="valor", columns="conjunto", values="media").round(3)
        display(tabla.rename(columns={"entrenamiento": "Entrenamiento", "test": "Test"}))

    c.dibujo_profundidad = _deslizador(widgets.IntSlider, "Niveles a dibujar", "Oculta lo que está más abajo.",
                                       value=3, min=1, max=10)

    def tarea_arbol():
        arbol = entrenar_arbol(c.dataset.value, params_c45(), c.semilla.value, c.test.value)
        nodos, hojas, profundidad = medir_arbol(arbol.raiz_)
        print(f"Nodos: {nodos} | Hojas: {hojas} | Profundidad: {profundidad}")
        mostrar_arbol(arbol.raiz_, c.dibujo_profundidad.value)

    pestanas = widgets.Tab([
        widgets.VBox([boton("comparar", "Comparar modelos", tarea_comparar), salidas["comparar"]]),
        widgets.VBox([c.barrido_modelo, c.barrido_parametro, c.barrido_valores,
                      boton("barrer", "Variar parámetro", tarea_barrer), salidas["barrer"]]),
        widgets.VBox([c.dibujo_profundidad, boton("arbol", "Dibujar árbol C4.5", tarea_arbol), salidas["arbol"]]),
    ])
    for indice, titulo in enumerate(("Comparar modelos", "Variar un parámetro", "Ver el árbol")):
        pestanas.set_title(indice, titulo)

    datos = _caja("Datos y evaluación", "Se aplican a las tres pestañas.",
                  [c.dataset, c.test, c.repeticiones, c.semilla])
    arbol = _caja("Árbol C4.5", "Implementación propia del equipo.",
                  [c.c45_limite, c.c45_profundidad, c.c45_minimo, c.c45_ganancia, c.c45_razon])
    bosque = _caja("Random Forest", "Modelo de scikit-learn.",
                   [c.rf_arboles, c.rf_atributos, c.rf_criterio, c.rf_bootstrap, c.rf_limite, c.rf_profundidad])
    vista = widgets.VBox([
        widgets.HTML("<h3 style='margin:0 0 8px'>C4.5 y Random Forest</h3>"),
        widgets.HBox([datos, arbol, bosque], layout=widgets.Layout(flex_flow="row wrap")),
        pestanas, estado,
    ])
    return SimpleNamespace(vista=vista, controles=c, botones=botones, salidas=salidas, estado=estado)


In [ ]:
interfaz = crear_interfaz()
display(interfaz.vista)
